# 04 — Supplementary Data-Quality and Leakage Checks

These checks back up Sections 3, 5 and 6 of the Interim Progress Report. They answer four questions the first three notebooks left open:

1. Do duplicate rows end up on both sides of the train/test split?
2. Do any identical flows carry different labels?
3. If we remove duplicates *before* splitting, do the scores drop? (If they do, duplicates were inflating them.)
4. How much does the model rely on raw port numbers?

We also compute the majority-class baseline, the simplest possible reference point.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "train_test_network.csv"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["label", "type", "src_ip", "dst_ip"])
row_hash = pd.util.hash_pandas_object(X, index=False)
print(df.shape)

(211043, 44)


## 1. Duplicates and conflicting labels

In [2]:
dup = df.duplicated()
print("Exact duplicate rows (all 44 columns):", int(dup.sum()), f"({dup.mean():.1%})")
print("\nDuplicates by category:")
print(df.loc[dup, "type"].value_counts())

print("\nRows whose 40 model features match another row:", int(X.duplicated(keep=False).sum()))
type_per_group = df.assign(h=row_hash).groupby("h")["type"].nunique()
label_per_group = df.assign(h=row_hash).groupby("h")["label"].nunique()
conflict_groups = type_per_group[type_per_group > 1].index
print("Identical-feature groups with different attack types:", len(conflict_groups),
      "| rows involved:", int(row_hash.isin(conflict_groups).sum()))
print("Identical-feature groups with different binary labels:", int((label_per_group > 1).sum()))

Exact duplicate rows (all 44 columns): 20569 (9.7%)

Duplicates by category:
type
normal        7960
ransomware    5265
xss           4863
backdoor      1289
dos           1008
password       139
injection       36
ddos             7
mitm             2
Name: count, dtype: int64



Rows whose 40 model features match another row: 37341
Identical-feature groups with different attack types: 11 | rows involved: 43
Identical-feature groups with different binary labels: 0


**Interpretation:** about one row in ten is an exact copy. A small number of identical flows carry different attack types, which is a sign of window-based labeling rather than per-flow inspection. The binary label never conflicts.

## 2. Do duplicates cross the train/test boundary?
Same 70/15/15 stratified split as notebooks 02 and 03.

In [3]:
ym = df["type"].astype(str)
X_train, X_temp, ym_train, ym_temp = train_test_split(X, ym, test_size=0.30, random_state=42, stratify=ym)
X_val, X_test, ym_val, ym_test = train_test_split(X_temp, ym_temp, test_size=0.50, random_state=42, stratify=ym_temp)

train_hashes = set(pd.util.hash_pandas_object(X_train, index=False))
in_train = pd.util.hash_pandas_object(X_test, index=False).isin(train_hashes).values
print(f"Test rows with an exact copy in training: {in_train.sum():,} ({in_train.mean():.1%})")
print(pd.Series(in_train, index=ym_test.index).groupby(ym_test).mean().round(3).sort_values(ascending=False).rename("share_with_copy_in_train"))

Test rows with an exact copy in training: 4,951 (15.6%)
type
xss           0.364
dos           0.348
ransomware    0.341
normal        0.159
scanning      0.094
backdoor      0.090
mitm          0.013
password      0.011
ddos          0.001
injection     0.001
Name: share_with_copy_in_train, dtype: float64


## 3. Majority-class baseline

In [4]:
yb_test = (ym_test != "normal").astype(int)
binary_guess = np.ones(len(yb_test), dtype=int)                # always "attack"
multi_guess = np.array(["normal"] * len(ym_test))               # always "normal"
baseline = pd.DataFrame([
    {"model": "majority_binary", "accuracy": accuracy_score(yb_test, binary_guess), "macro_f1": f1_score(yb_test, binary_guess, average="macro")},
    {"model": "majority_multiclass", "accuracy": accuracy_score(ym_test, multi_guess), "macro_f1": f1_score(ym_test, multi_guess, average="macro")},
])
baseline

,model,accuracy,macro_f1
0,majority_binary,0.763086,0.432813
1,majority_multiclass,0.236914,0.038307


## 4. Retrain after de-duplicating *before* the split
Drop exact feature duplicates (keep the first) and the rows in conflicting groups, then repeat the same split, preprocessing and models.

In [5]:
keep = (~row_hash.duplicated()) & (~row_hash.isin(conflict_groups))
d = df[keep.values].reset_index(drop=True)
Xd = X[keep.values].reset_index(drop=True)
yd = d["type"].astype(str)
print("Rows after de-duplication:", len(d))

Xtr, Xtmp, ytr, ytmp = train_test_split(Xd, yd, test_size=0.30, random_state=42, stratify=yd)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, random_state=42, stratify=ytmp)

cat_cols = Xtr.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in Xtr.columns if c not in cat_cols]
pre = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
A = pre.fit_transform(Xtr)
B = pre.transform(Xte)

rows = []
for name, model, y_fit, y_eval in [
    ("random_forest_binary_dedup", RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42, n_jobs=-1), (ytr != "normal").astype(int), (yte != "normal").astype(int)),
    ("random_forest_multiclass_dedup", RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42, n_jobs=-1), ytr, yte),
    ("logistic_regression_multiclass_dedup", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), ytr, yte),
]:
    model.fit(A, y_fit)
    pred = model.predict(B)
    rep = classification_report(y_eval, pred, output_dict=True, zero_division=0)
    rows.append({"model": name, "accuracy": accuracy_score(y_eval, pred), "macro_f1": f1_score(y_eval, pred, average="macro"),
                 "mitm_precision": rep.get("mitm", {}).get("precision", np.nan), "mitm_recall": rep.get("mitm", {}).get("recall", np.nan),
                 "mitm_f1": rep.get("mitm", {}).get("f1-score", np.nan)})
dedup_results = pd.DataFrame(rows)
dedup_results

Rows after de-duplication: 183570


,model,accuracy,macro_f1,mitm_precision,mitm_recall,mitm_f1
0,random_forest_binary_dedup,0.999419,0.999176,NaN,NaN,NaN
1,random_forest_multiclass_dedup,0.988778,0.968417,0.687500,0.916667,0.785714
2,logistic_regression_multiclass_dedup,0.863996,0.826659,0.436047,0.961538,0.600000


**Interpretation:** compare these with `results/metrics/model_comparison.csv`. The scores barely change, so duplicates are **not** what makes the Random Forest look so strong. The more likely explanation is that each attack type comes from only a few attacker machines in one testbed. The planned host-grouped split tests this.

## 5. How much does the model rely on port numbers?
A quick ablation with ordinal-encoded categoricals and an 85/15 split on the de-duplicated data.

In [6]:
Xo = Xd.copy()
Xo[cat_cols] = OrdinalEncoder().fit_transform(Xo[cat_cols].astype(str))
Xa, Xb, ya, yb_ = train_test_split(Xo, yd, test_size=0.15, random_state=42, stratify=yd)
ablation = []
for dropped in ([], ["src_port", "dst_port"]):
    rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(Xa.drop(columns=dropped), ya)
    pred = rf.predict(Xb.drop(columns=dropped))
    rep = classification_report(yb_, pred, output_dict=True, zero_division=0)
    ablation.append({"ports_dropped": bool(dropped), "macro_f1": f1_score(yb_, pred, average="macro"), "mitm_f1": rep["mitm"]["f1-score"]})
    if not dropped:
        print(pd.Series(rf.feature_importances_, Xa.columns).sort_values(ascending=False).head(10).round(3))
pd.DataFrame(ablation)

src_ip_bytes    0.162
dst_port        0.110
duration        0.108
src_port        0.106
conn_state      0.093
dst_ip_bytes    0.074
src_bytes       0.073
src_pkts        0.063
dst_pkts        0.048
dst_bytes       0.043
dtype: float64


,ports_dropped,macro_f1,mitm_f1
0,False,0.965885,0.767442
1,True,0.953201,0.766467


**Interpretation:** ports matter (about a fifth of total importance) but dropping them costs only about one point of macro-F1, so bucketing them is a cheap safeguard against memorizing lab-specific ports.

In [7]:
pd.concat([baseline, dedup_results], ignore_index=True).to_csv(METRICS_DIR / "supplementary_checks.csv", index=False)
pd.DataFrame(ablation).to_csv(METRICS_DIR / "port_ablation.csv", index=False)
print("Saved supplementary_checks.csv and port_ablation.csv")

Saved supplementary_checks.csv and port_ablation.csv
